# Stage 1 — Non-Instruction Fine-Tuning

**Domain:** Customer Support Assistant
**Base model:** `unsloth/Qwen2.5-0.5B` (plain base checkpoint)

This notebook adapts the base model to customer-support *language and terminology* by training it as a plain causal language model on raw domain text (`data/non_instruction_data.txt`) — no instructions, no Q&A framing, just next-token prediction on knowledge-base-style prose about refunds, order tracking, cancellations, payments, etc.

Run this notebook on a GPU runtime (Google Colab's free T4 is enough for a 0.5B model in 4-bit).


## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth


## 2. Load the base model with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None          # auto-detect (bfloat16 on Ampere+, else float16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(model.config._name_or_path)


## 3. Load, clean and chunk the raw domain text

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/non_instruction_data.txt")
raw_text = DATA_PATH.read_text(encoding="utf-8")

# The file is already one clean paragraph per blank-line-separated block
# (see scripts/build_datasets.py) - split, strip, drop anything empty.
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0][:300], "...")


In [ ]:
def chunk_paragraphs(paragraphs, tokenizer, block_size=512):
    """Greedily pack paragraphs into ~block_size-token chunks so each
    training example is a reasonably dense block of raw domain text rather
    than one short paragraph per row."""
    chunks, buffer = [], ""
    for p in paragraphs:
        candidate = (buffer + "\n\n" + p).strip() if buffer else p
        if buffer and len(tokenizer(candidate)["input_ids"]) > block_size:
            chunks.append(buffer)
            buffer = p
        else:
            buffer = candidate
    if buffer:
        chunks.append(buffer)
    return chunks


chunks = chunk_paragraphs(paragraphs, tokenizer, block_size=512)
print(f"Built {len(chunks)} training chunks from {len(paragraphs)} paragraphs")


In [ ]:
from datasets import Dataset

raw_dataset = Dataset.from_dict({"text": chunks})
raw_dataset


## 4. Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
model.print_trainable_parameters()


## 5. Train on raw domain text (plain causal-LM objective, no instruction formatting)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = raw_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "../outputs/stage1_checkpoints",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


## 6. Save the Stage 1 adapter

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)
model.save_pretrained("../outputs/stage1_adapter")
tokenizer.save_pretrained("../outputs/stage1_adapter")
print("Saved Stage 1 non-instruction adapter to ../outputs/stage1_adapter")


## 7. Quick sanity test after Stage 1

In [ ]:
FastLanguageModel.for_inference(model)

for prompt in ["Our refund policy", "To track your order,", "If your payment failed,"]:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=60, use_cache=True, do_sample=False)
    print("PROMPT:", prompt)
    print("CONTINUATION:", tokenizer.decode(output[0], skip_special_tokens=True))
    print("---")


## Next step

Continue with `instruction_finetuning.ipynb`. It will detect `../outputs/stage1_adapter` and resume from it automatically before Stage 2 (instruction) fine-tuning.
